# Notebook 1: Index Vector Podcast Transcripts into OpenSearch

In this notebook you will:
1. Connect to a local OpenSearch cluster
2. Create a **k-NN index** with cosine-similarity vector search
3. Parse 33 podcast episode transcripts
4. Generate **sentence embeddings** (384-dim, all-MiniLM-L6-v2)
5. Bulk-index all chunks, ready for semantic search\!

---
### Prerequisites
```bash
# 1. Start OpenSearch
cd workshop/
docker compose up -d

# 2. Create venv and install Python deps
uv sync
```
Check OpenSearch is up: http://localhost:9200

## 0 · Verify OpenSearch is reachable

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.opensearch_client import get_client

client = get_client()
info = client.info()
print(f"✅ Connected to OpenSearch {info['version']['number']}")
print(f"   Cluster: {info['cluster_name']}")

✅ Connected to OpenSearch 2.13.0
   Cluster: docker-cluster


## 1 · Create the k-NN Index

We create an index with two key features:
- **`knn_vector` field:** stores 384-dim embeddings using HNSW graph
- **`text` field:**  keeps original transcript text for BM25 lexical search (used in Notebook 2)

In [2]:
from src.opensearch_client import create_index, INDEX_SETTINGS, INDEX_NAME
import json

# Show the index mapping so you can see what we're creating
print("Index mapping:")
print(json.dumps(INDEX_SETTINGS['mappings']['properties']['embedding'], indent=2))
print()

# Create the index (set recreate=True to wipe and rebuild)
create_index(client, recreate=False)

Index mapping:
{
  "type": "knn_vector",
  "dimension": 384,
  "method": {
    "name": "hnsw",
    "space_type": "cosinesimil",
    "engine": "nmslib",
    "parameters": {
      "ef_construction": 128,
      "m": 24
    }
  }
}

Index 'vector-podcast' already exists, skipping creation


## 2 · Parse Podcast Markdown Files

Each file has:
- **YAML frontmatter**: title, description, pub_date, url  
- **Body**: Whisper-generated transcript (raw speech, unedited)

We chunk the transcript into ~400-word windows with 50-word overlap so long episodes stay searchable.

In [8]:
from src.parser import load_podcast_chunks

DATA_DIR = os.path.join(os.getcwd(), '..', 'vector-podcast')

chunks = load_podcast_chunks(DATA_DIR, chunk_size=400, overlap=50)

print(f"Loaded {len(chunks)} chunks from {len(set(c.episode_title for c in chunks))} episodes")
print()

# Preview one chunk
sample = chunks[5]
print(f"Episode : {sample.episode_title}")
print(f"Chunk   : {sample.chunk_index + 1} of {sample.total_chunks}")
print(f"Text    : {sample.chunk_text[:300]}...")

Loaded 1018 chunks from 33 episodes

Episode : Adding ML layer to Search: Hybrid Search Optimizer with Daniel Wrigley and Eric Pugh
Chunk   : 6 of 26
Text    : the best algorithm it's just as fast as the one that we use uh vectors is sort of this idea of there are richer ways of understanding user queries and the content tech and just going beyond taxes the you know it's absolutely wonderful right lots of different things I mean some point we'll do a vecto...


## 3 · Generate Embeddings

We use **`all-MiniLM-L6-v2`**, a fast, lightweight model that produces 384-dimensional semantic vectors.
It's ideal for workshops: downloads ~90 MB and runs on CPU in seconds per batch.

| Model | Dims | Speed | Quality |
|---|---|---|---|
| all-MiniLM-L6-v2 | 384 | ⚡⚡⚡ | Good |
| all-mpnet-base-v2 | 768 | ⚡⚡ | Better |
| text-embedding-3-large | 3072 | API call | Best |

In [4]:
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np

print("Loading embedding model (downloads ~90 MB on first run)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✅ Model loaded, output dim: {model.get_sentence_embedding_dimension()}")

Loading embedding model (downloads ~90 MB on first run)...
✅ Model loaded, output dim: 384


In [5]:
# Embed all chunks in batches
texts = [c.chunk_text for c in chunks]

print(f"Embedding {len(texts)} chunks...")
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,   # unit vectors → cosine sim = dot product
    convert_to_numpy=True,
)

print(f"\n✅ Embeddings shape: {embeddings.shape}")
print(f"   Norm of first vector: {np.linalg.norm(embeddings[0]):.4f}  (should be ~1.0)")

Embedding 1018 chunks...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]


✅ Embeddings shape: (1018, 384)
   Norm of first vector: 1.0000  (should be ~1.0)


## 4 · Bulk-Index into OpenSearch

We use `helpers.bulk()` for efficient batch ingestion as it's far faster than indexing one document at a time.

In [6]:
from src.opensearch_client import bulk_index

embedding_lists = embeddings.tolist()  # numpy → plain Python lists for JSON serialisation

print("Indexing into OpenSearch...")
indexed = bulk_index(client, chunks, embedding_lists, batch_size=100)

print(f"\n✅ Indexed {indexed} documents")

# Wait for refresh so documents are immediately searchable
client.indices.refresh(INDEX_NAME)
count = client.count(index=INDEX_NAME)['count']
print(f"   Documents in index: {count}")

Indexing into OpenSearch...

✅ Indexed 1018 documents
   Documents in index: 1018


## 5 · Quick Sanity Check

Let's run a quick k-NN query to make sure everything is working before moving to the RAG notebook.

In [7]:
from src.opensearch_client import knn_search

test_query = "What is HNSW and how does it work?"
test_embedding = model.encode(test_query, normalize_embeddings=True).tolist()

results = knn_search(client, test_embedding, k=3)

print(f"Query: '{test_query}'\n")
for i, r in enumerate(results, 1):
    print(f"  [{i}] Score: {r['score']:.4f} | {r['episode_title']}")
    print(f"      {r['chunk_text'][:200]}...")
    print()

Query: 'What is HNSW and how does it work?'

  [1] Score: 0.5959 | Yury Malkov - Staff Engineer, Twitter - Author of the most adopted ANN algorithm HNSW
      Yeah it kind of brings transparency at least with the process and also as you mentioned someone can learn how to do these things right so I think it's also useful and maybe it prevents situations when...

  [2] Score: 0.5873 | Joan Fontanals - Principal Engineer - Jina AI
      also python? I don't know if we are for instance I think we are also using some bindings for HLW so you are using probably C++ version of HNSW binding to python right? yes that's for sure but I don't ...

  [3] Score: 0.5852 | Yury Malkov - Staff Engineer, Twitter - Author of the most adopted ANN algorithm HNSW
      at the same time. Yeah so that that's why I ignore some of the stuff there's also a focus on like custom distances so even though the hnsw lip supports only free distances is pretty easy to implement ...



## ✅ Done!

You have successfully:
- Created an OpenSearch k-NN index with HNSW + cosine similarity
- Parsed and chunked 33 podcast episode transcripts  
- Generated 384-dimensional sentence embeddings  
- Bulk-indexed everything into OpenSearch

**→ Open `02_rag_pipeline.ipynb` to build the RAG pipeline!**